In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from lifelines import WeibullAFTFitter, KaplanMeierFitter

In [ ]:
hollymac_data = pd.read_excel('HollyMcNamara_GoalsBySeason.xlsx')

In [ ]:
def season_to_decimal(season_str):
    start_year = int(season_str.split('-')[0])
    return start_year + 0.5

In [ ]:
hollymac_data['season_decimal'] = hollymac_data['season'].apply(season_to_decimal)

In [ ]:
milestone_season = 2024.5
hollymac_data = hollymac_data.sort_values('season_decimal')

In [ ]:
hollymac_data['cumulative_matches'] = hollymac_data['matches'].cumsum()
hollymac_data['cumulative_goals'] = hollymac_data['goals'].cumsum()

In [ ]:
debut_season = hollymac_data['season_decimal'].min()

In [ ]:
hollymac_data['duration'] = hollymac_data['season_decimal'] - debut_season + 1

In [ ]:
hollymac_data['event'] = 1

In [ ]:
survival_data = hollymac_data[['duration', 'event', 'cumulative_matches', 'cumulative_goals']]

In [ ]:
aft = WeibullAFTFitter()
aft.fit(survival_data, duration_col = 'duration', event_col = 'event')

In [ ]:
print("AFT Model Summary:\n")
print(aft.summary)

In [ ]:
median_duration = aft.predict_median(survival_data.iloc[[-1]])[0]
predicted_season_decimal = debut_season + median_duration - 1

In [ ]:
def decimal_to_season(dec):
    start_year = int(np.floor(dec))
    if dec - start_year >= 0.5:
        return f"{start_year} - {str(start_year + 1)[2:]}"
    else:
        return str(start_year)

In [ ]:
predicted_season = decimal_to_season(predicted_season_decimal)
print(f"\nPredicted median season for first international goal: {predicted_season}")

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(survival_data['duration'], event_observed = survival_data['event'])

In [ ]:
kmf.plot_survival_function()
plt.title("Kaplan-Meier estimation of probability of no international goals yet")
plt.xlabel("Seasons since club debut (decimal)")
plt.ylabel("Survival probability")
plt.show()